In [1]:
# %%
# =============================================================================
# 셀 1. 라이브러리 및 환경 설정
# =============================================================================
# 필요 패키지 설치 (최초 1회)
# pip install chromadb pypdf langchain langchain-openai langchain-community tiktoken

import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# LangChain / ChromaDB
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 저장 경로 생성
os.makedirs('../results/tables', exist_ok=True)
os.makedirs('../results/rag_inference', exist_ok=True)
os.makedirs('../data/chroma_db', exist_ok=True)

# API 설정
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')
EMBED_MODEL    = 'text-embedding-3-small'

if not OPENAI_API_KEY:
    raise ValueError(
        "\n[ERROR] OPENAI_API_KEY가 설정되지 않았습니다.\n"
        "  .env 파일에 OPENAI_API_KEY=sk-... 입력하세요."
    )

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"[INFO] LLM Model   : {LLM_MODEL}")
print(f"[INFO] Embed Model : {EMBED_MODEL}")


# %%
# =============================================================================
# 셀 2. PDF 로드 및 청크 분할
# =============================================================================
PDF_SOURCES = {
    'law'    : '../data/source_law.pdf',
    'manual' : '../data/source_manual.pdf',
}

# PDF 로드
all_docs = []
for src_name, pdf_path in PDF_SOURCES.items():
    loader = PyPDFLoader(pdf_path)
    pages  = loader.load()
    for p in pages:
        p.metadata['source_type'] = src_name   # 출처 태그 부여
    all_docs.extend(pages)
    print(f"[INFO] {src_name}: {len(pages)}페이지 로드")

print(f"[INFO] 전체 페이지 합계: {len(all_docs)}페이지")

# 청크 분할
# - chunk_size=800  : 법령 조문 1~2개 수준, 맥락 보존
# - chunk_overlap=100: 조문 경계 손실 방지
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", "제", "①", "②", "③", "。", " ", ""]
)
chunks = splitter.split_documents(all_docs)
print(f"[INFO] 청크 수: {len(chunks)}개")
print(f"[INFO] 청크 평균 길이: {np.mean([len(c.page_content) for c in chunks]):.0f}자")

# 청크 샘플 확인
print("\n[샘플 청크 - law]")
law_chunks = [c for c in chunks if c.metadata.get('source_type') == 'law']
print(law_chunks[0].page_content[:300])


# %%
# =============================================================================
# 셀 3. ChromaDB 벡터스토어 구축
# =============================================================================
CHROMA_DIR = '../data/chroma_db'
COLLECTION = 'nkrefugee_legal'

embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    openai_api_key=OPENAI_API_KEY
)

# 기존 컬렉션 있으면 재사용, 없으면 신규 생성
if os.path.exists(os.path.join(CHROMA_DIR, 'chroma.sqlite3')):
    print("[INFO] 기존 ChromaDB 로드")
    vectorstore = Chroma(
        collection_name=COLLECTION,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR
    )
    print(f"[INFO] 기존 문서 수: {vectorstore._collection.count()}개")
else:
    print("[INFO] ChromaDB 신규 구축 중... (수 분 소요)")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION,
        persist_directory=CHROMA_DIR
    )
    print(f"[INFO] 임베딩 완료: {vectorstore._collection.count()}개 청크 저장")

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}   # 상위 5개 청크 검색
)
print("[INFO] Retriever 준비 완료")


# %%
# =============================================================================
# 셀 4. 검색 품질 사전 점검 (spot-check)
# =============================================================================
TEST_QUERIES = [
    "북한이탈주민이 보호결정을 받으려면 어떻게 해야 하나요?",
    "마약 소지로 체포된 경우 어떤 절차를 밟아야 하나요?",
    "정착지원금은 얼마나 받을 수 있나요?",
]

print("[INFO] 검색 품질 점검\n")
for q in TEST_QUERIES:
    docs = retriever.invoke(q)
    print(f"[질문] {q}")
    for i, d in enumerate(docs[:2]):   # 상위 2개만 출력
        src = d.metadata.get('source_type', 'unknown')
        pg  = d.metadata.get('page', '?')
        print(f"  [{i+1}] ({src}, p.{pg}) {d.page_content[:120]}...")
    print()


# %%
# =============================================================================
# 셀 5. 시스템 프롬프트 정의 (RAG 버전)
# =============================================================================
SYSTEM_PROMPT_RAG = """당신은 사회적 취약계층을 위한 법률지원 AI 어시스턴트입니다.
현재는 북한이탈주민의 법률 문제를 중심으로 안내하고 있습니다.

[응답 원칙]
1. 반드시 아래 [참고 문서]에 근거하여 답변하십시오.
   참고 문서에 없는 내용은 "관련 법령 확인이 필요합니다"라고 명시하십시오.
2. 법적 근거(법령명 및 조항)를 반드시 명시하십시오.
   예시: 「북한이탈주민의 보호 및 정착지원에 관한 법률」 제00조
3. 고위험 상황(형사처벌, 보호종료, 신변위협, 가정폭력 등)에서는
   반드시 전문 법률기관 연락처를 안내하십시오.
   - 대한법률구조공단: 132
   - 남북하나재단: 1577-6635
   - 경찰: 112
   - 가정폭력 상담: 1366
4. AI의 응답은 법률 상담을 대체하지 않으며, 최종 판단은
   반드시 변호사 등 전문가와 상담하시기 바랍니다.
5. 명확하고 이해하기 쉬운 언어로 답변하십시오.

[참고 문서]
{context}
"""


# %%
# =============================================================================
# 셀 6. RAG 응답 생성 함수
# =============================================================================
def retrieve_context(question: str, k: int = 5) -> tuple[str, list]:
    """
    질문과 관련된 청크를 검색하여 컨텍스트 문자열과 메타데이터를 반환합니다.

    Returns
    -------
    context_str : 프롬프트에 삽입할 텍스트
    source_meta : 검색된 청크의 메타데이터 리스트
    """
    docs = retriever.invoke(question)
    context_parts = []
    source_meta   = []
    for i, d in enumerate(docs):
        src = d.metadata.get('source_type', 'unknown')
        pg  = d.metadata.get('page', '?')
        context_parts.append(
            f"[출처: {'법령' if src=='law' else '매뉴얼'}, p.{pg}]\n{d.page_content}"
        )
        source_meta.append({'source_type': src, 'page': pg})
    return "\n\n---\n\n".join(context_parts), source_meta


def generate_rag_response(question: str,
                          model: str = LLM_MODEL,
                          max_tokens: int = 1000,
                          retry: int = 3,
                          sleep_sec: float = 1.0) -> dict:
    """
    RAG 파이프라인: 검색 → 프롬프트 구성 → LLM 응답 생성

    Returns
    -------
    dict : {response, context, source_meta, input_tokens, output_tokens, error}
    """
    # 1. 컨텍스트 검색
    context_str, source_meta = retrieve_context(question)

    # 2. 프롬프트 구성
    system_filled = SYSTEM_PROMPT_RAG.format(context=context_str)

    # 3. LLM 호출
    for attempt in range(retry):
        try:
            res = client.chat.completions.create(
                model=model,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_filled},
                    {"role": "user",   "content": question}
                ]
            )
            time.sleep(sleep_sec)
            return {
                "response"      : res.choices[0].message.content.strip(),
                "context"       : context_str,
                "source_meta"   : json.dumps(source_meta, ensure_ascii=False),
                "input_tokens"  : res.usage.prompt_tokens,
                "output_tokens" : res.usage.completion_tokens,
                "error"         : None
            }
        except Exception as e:
            print(f"  [WARN] 시도 {attempt+1}/{retry} 실패: {e}")
            time.sleep(sleep_sec * (attempt + 1))

    return {
        "response"      : None,
        "context"       : context_str,
        "source_meta"   : json.dumps(source_meta, ensure_ascii=False),
        "input_tokens"  : 0,
        "output_tokens" : 0,
        "error"         : "API 호출 실패"
    }


# %%
# =============================================================================
# 셀 7. 전체 문항 RAG 추론 실행 (체크포인트 포함)
# =============================================================================
CHECKPOINT_RAG  = '../results/rag_inference/checkpoint_rag.json'
RESULT_RAG_PATH = '../results/rag_inference/rag_responses.csv'

# 데이터 로드
with open('../data/nkrefugee_qa_1_100.json', 'r', encoding='utf-8') as f:
    qa_1_100 = json.load(f)
with open('../data/nkrefugee_qa_101_200.json', 'r', encoding='utf-8') as f:
    qa_101_200 = json.load(f)
df = pd.DataFrame(qa_1_100 + qa_101_200)
print(f"[INFO] 총 {len(df)}개 문항")

# 체크포인트 로드
if os.path.exists(CHECKPOINT_RAG):
    with open(CHECKPOINT_RAG, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)
    print(f"[INFO] 체크포인트 발견: {len(checkpoint)}개 완료")
else:
    checkpoint = {}
    print("[INFO] 처음부터 시작")

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="RAG 추론"):
    qid = row['id']

    if qid in checkpoint:
        results.append(checkpoint[qid])
        continue

    out = generate_rag_response(row['question'])

    record = {
        "id"            : qid,
        "category"      : row['category'],
        "difficulty"    : row['difficulty'],
        "risk_level"    : row['risk_level'],
        "source"        : row['source'],
        "legal_basis"   : row['legal_basis'],
        "question"      : row['question'],
        "ground_truth"  : row['ground_truth'],
        "rag_response"  : out['response'],
        "context"       : out['context'],
        "source_meta"   : out['source_meta'],
        "input_tokens"  : out['input_tokens'],
        "output_tokens" : out['output_tokens'],
        "error"         : out['error']
    }

    results.append(record)
    checkpoint[qid] = record

    if len(checkpoint) % 10 == 0:
        with open(CHECKPOINT_RAG, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)

# 최종 저장
with open(CHECKPOINT_RAG, 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, ensure_ascii=False, indent=2)

df_rag = pd.DataFrame(results)
df_rag.to_csv(RESULT_RAG_PATH, index=False, encoding='utf-8-sig')
print(f"\n[INFO] RAG 추론 완료: {len(df_rag)}개")
print(f"[INFO] 저장: {RESULT_RAG_PATH}")


# %%
# =============================================================================
# 셀 8. 추론 결과 기본 검증
# =============================================================================
df_rag = pd.read_csv(RESULT_RAG_PATH, encoding='utf-8-sig')

n_error = df_rag['error'].notna().sum()
print(f"[INFO] 오류 건수: {n_error}개")

df_rag['response_len'] = df_rag['rag_response'].fillna('').apply(len)
print(f"\n[INFO] 응답 길이 통계")
print(df_rag['response_len'].describe().round(1))

total_in  = df_rag['input_tokens'].sum()
total_out = df_rag['output_tokens'].sum()
print(f"\n[INFO] 총 입력 토큰 : {total_in:,}")
print(f"[INFO] 총 출력 토큰 : {total_out:,}")
print(f"[INFO] 총 토큰      : {total_in + total_out:,}")


# %%
# =============================================================================
# 셀 9. RAG vs 기존(프롬프트 전용) 응답 길이 비교
# =============================================================================
BASE_RESULT_PATH = '../results/inference/llm_responses.csv'

if os.path.exists(BASE_RESULT_PATH):
    df_base = pd.read_csv(BASE_RESULT_PATH, encoding='utf-8-sig')
    df_base['response_len'] = df_base['llm_response'].fillna('').apply(len)

    compare = pd.DataFrame({
        'Baseline (Prompt-only)' : df_base['response_len'],
        'RAG'                    : df_rag['response_len']
    })
    print("[INFO] 응답 길이 비교 (문자 수)")
    print(compare.describe().round(1))
else:
    print("[WARN] 기존 추론 결과 없음 — 03_evaluation_g1_g2.ipynb 먼저 실행 필요")


# %%
# =============================================================================
# 셀 10. 고위험 샘플 확인
# =============================================================================
df_high = df_rag[df_rag['risk_level'] == 'high'].reset_index(drop=True)
print(f"[INFO] 고위험 문항 수: {len(df_high)}개\n")

for i, row in df_high.head(2).iterrows():
    print(f"{'='*60}")
    print(f"[{row['id']}] {row['category']} | {row['difficulty']}")
    print(f"[질문] {row['question']}")
    print(f"[RAG 응답]\n{row['rag_response']}")
    print(f"[검색된 소스]\n{row['source_meta']}")
    print()


# %%
# =============================================================================
# 셀 11. 최종 요약
# =============================================================================
print("=" * 50)
print("02b_rag_pipeline 완료 요약")
print("=" * 50)
print(f"총 추론 문항   : {len(df_rag)}개")
print(f"성공           : {len(df_rag) - n_error}개")
print(f"오류           : {n_error}개")
print(f"총 사용 토큰   : {total_in + total_out:,}")
print(f"ChromaDB 위치  : ../data/chroma_db/")
print()
print("[저장 완료]")
print(f"  rag_inference/ : rag_responses.csv")
print(f"  rag_inference/ : checkpoint_rag.json")
print()
print("[다음 단계] 03_evaluation_g1_g2.ipynb 에서")
print("  llm_responses.csv  (Baseline)")
print("  rag_responses.csv  (RAG)")
print("  두 파일을 함께 평가하여 비교 분석 진행")

[INFO] LLM Model   : gpt-4o-mini
[INFO] Embed Model : text-embedding-3-small
[INFO] law: 36페이지 로드
[INFO] manual: 179페이지 로드
[INFO] 전체 페이지 합계: 215페이지
[INFO] 청크 수: 393개
[INFO] 청크 평균 길이: 585자

[샘플 청크 - law]
북 한 이 탈 주 민 의  보호  및  정 착 지 원에  관한  법 률
시 행  2025. 10. 9. 법 률  제  20906 호2025. 4. 8. 일부개정통 일부
제 1 조 ( 목적 )
[ 전문개정  2010. 3. 26.]
이 법 은  군 사 분계 선  이 북 지 역 에서  벗 어나  대한 민 국 의  보호를  받 으 려 는  군 사 분계 선  이 북 지 역 의  주 민 이 정치 , 경제 , 사
회 , 문화  등  모 든  생 활  영 역 에서  신 속 히  적 응 ㆍ 정 착 하 는  데  필 요한  보호  및  지 원에  
[INFO] ChromaDB 신규 구축 중... (수 분 소요)
[INFO] 임베딩 완료: 393개 청크 저장
[INFO] Retriever 준비 완료
[INFO] 검색 품질 점검

[질문] 북한이탈주민이 보호결정을 받으려면 어떻게 해야 하나요?
  [1] (manual, p.39) 에 청구하도록 요청하여야 합니다.
나. 보호결정 효과
(1) 보호대상자
북한이탈주민 보호결정을 한 때에는 정착시설장은 보호대상자의 등록기준지, 가족관계, 
경력 등 필요한 사항을 기록한 등록대장을 관리･보존합니다(법...
  [2] (manual, p.3) 2. 북한이탈주민 보호 결정 ·····················································································24
가. 북한이탈주민 보호 결정 ...

[질문] 마약 소지로 체포된 경우 어떤 절차를 밟아야 하나요?
  [1] (manual, p.160) Ⅷ. 법률구조 및 법

RAG 추론: 100%|██████████████████████████████████████████████████████████████████████| 200/200 [24:38<00:00,  7.39s/it]



[INFO] RAG 추론 완료: 200개
[INFO] 저장: ../results/rag_inference/rag_responses.csv
[INFO] 오류 건수: 0개

[INFO] 응답 길이 통계
count    200.0
mean     423.6
std      155.2
min       77.0
25%      313.0
50%      403.5
75%      524.2
max      898.0
Name: response_len, dtype: float64

[INFO] 총 입력 토큰 : 428,025
[INFO] 총 출력 토큰 : 47,092
[INFO] 총 토큰      : 475,117
[INFO] 응답 길이 비교 (문자 수)
       Baseline (Prompt-only)    RAG
count                   200.0  200.0
mean                    517.7  423.6
std                     115.9  155.2
min                     231.0   77.0
25%                     434.0  313.0
50%                     514.5  403.5
75%                     596.2  524.2
max                     829.0  898.0
[INFO] 고위험 문항 수: 55개

[Q001] 보호결정·국적 | 기본
[질문] 북한에서 탈출해서 중국에 있는데, 한국에 오려면 어떻게 해야 하나요?
[RAG 응답]
북한에서 탈출하여 중국에 체류하고 있는 북한이탈주민이 한국에 입국하기 위해서는 다음과 같은 절차를 거쳐야 합니다.

1. **탈북 브로커 이용**: 대다수 북한이탈주민은 탈북 브로커를 통해 한국에 입국합니다. 이들은 중국에서 북한이탈주민을 남한으로 입국시키는 역할을 합니다. 이 과정에서 직접 계약형으로서 북한이탈주민이 용역 제공자와 계약을 통해 한국에 도착하기 위한